In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module=r'matplotlib\.image')
import logging
logging.getLogger('matplotlib.mathtext').setLevel(logging.WARNING)

import sys, os
sys.path.append(os.path.abspath("light_sbb/"))
sys.path.append(os.path.abspath("light_sbb/alae/"))
sys.path.append(os.path.abspath("light_sbb/alae/training_artifacts/"))

import torch
import numpy as np
from matplotlib import pyplot as plt

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown
import dlutils

if not os.path.isdir('data'):
    os.makedirs('data')

urls = {
    "data/age.npy": "https://drive.google.com/uc?id=1Vi6NzxCsS23GBNq48E-97Z9UuIuNaxPJ",
    "data/gender.npy": "https://drive.google.com/uc?id=1SEdsmQGL3mOok1CPTBEfc_O1750fGRtf",
    "data/latents.npy": "https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8",
    "data/test_images.npy": "https://drive.google.com/uc?id=1SjBWWlPjq-dxX4kxzW-Zn3iUR3po8Z0i",
}

if not all(os.path.isfile(path) for path in urls.keys()):
    for name, url in urls.items():
        gdown.download(url, os.path.join(f"{name}"), quiet=False)

In [ ]:
from alae_ffhq_inference import load_model, decode, encode
from light_sbb.utils import TensorSampler, sample_alae, sample_alae_beta_large  # !pip install POT
from alae_utils import plot_alae
from light_sbb.lightsbm import LightSBM
from light_sbb.lightsbm import MLP_network 
%matplotlib inline

In [ ]:
latents = np.load("data/latents.npy")
gender = np.load("data/gender.npy")
age = np.load("data/age.npy")
test_inp_images = np.load("data/test_images.npy")

In [ ]:
latents.shape, gender.shape, age.shape, test_inp_images.shape

In [ ]:
path = 'light_sbb/alae/training_artifacts/ffhq/model_157.pth'
if not os.path.isfile(path):
     dlutils.download.from_url(
        'https://alaeweights.s3.us-east-2.amazonaws.com/ffhq/model_157.pth',
        directory=os.path.dirname(path)
    )

In [ ]:
alae_model = load_model("light_sbb/alae/ffhq.yaml",
                        training_artifacts_dir="light_sbb/alae/training_artifacts/ffhq/")

In [ ]:
# TRAINING
%run C:/Users/alexa/icml_2026/light_sbb/run_alae.py  

In [ ]:
dim = 512
eps = 0.1
n_potentials = 10
S_init = 0.1
beta = 1

path = f"model/b{beta}_e{eps}.pkl" 

train_size = 60000
test_size = 10000

train_latents, test_latents = latents[:train_size], latents[train_size:]
train_gender, test_gender = gender[:train_size], gender[train_size:]
train_age, test_age = age[:train_size], age[train_size:]

x_inds_test = np.arange(test_size)[
    (test_age >= 18).reshape(-1)*(test_age != -1).reshape(-1)
]

model = LightSBM(dim=dim,
                 n_potentials=n_potentials,
                 epsilon=eps,
                 S_diagonal_init=S_init,
                 is_diagonal=True)

with open(path, "rb") as f:
    state_dict = pickle.load(f)
model.load_state_dict(state_dict)

if beta < 100:
    model_inv = MLP_network(input_dim=dim,
                            t_model=32,
                            d_model=128)
    
    path_inv = f"model_inv/b{beta}_e{eps}.pkl"
    
    with open(path_inv, "rb") as f:
        state_dict_inv = pickle.load(f)
    model_inv.load_state_dict(state_dict_inv)

In [ ]:
y_decoded_all, x_decoded_all, inp_images = sample_alae(model, model_inv, alae_model, beta, x_inds_test, test_inp_images,
                                                           test_latents, 3, device, SEED=42, n_pictures=10)  

In [ ]:
%matplotlib inline
plot_alae(x_decoded_all, n_pictures=10, number_of_samples=3, inp_images=inp_images, path=None)